# Phase 1 + 1B + 1C -- Graph Engine: GNN vs Tabular Baseline on Fraud Detection

Reproduces the comparison defined in `docs/PHASE_1_GRAPH_ENGINE.md`, then extends it per `docs/PHASE_1B_GRAPH_ENGINE_DEPTH.md` and `docs/PHASE_1C_GRAPH_ENGINE_ADVANCED.md`:

**Phase 1**
1. Load the Elliptic Bitcoin dataset via PyTorch Geometric's built-in loader.
2. EDA: label distribution, degree distribution, time-step distribution.
3. Define a temporal split (train on time steps 1-30, test on 36-49, 31-35 as a validation buffer), excluding unlabeled ("unknown") nodes.
4. Train a tuned XGBoost baseline on raw node features (no graph structure).
5. Train a 2-layer GraphSAGE model on the same temporal split.
6. Evaluate the trained GraphSAGE model a second way -- on a random split of the labeled nodes with the same proportions -- to quantify the temporal-vs-random generalization gap.

**Phase 1B (depth)**
7. Convergence check on the default GraphSAGE run.
8. A fair Optuna tuning pass for GraphSAGE, with early stopping on a validation slice (time steps 31-35).
9. A GAT model, plus an attention-based "why was this flagged" example for one correctly-flagged illicit node.
10. A Tier-1+Tier-2 XGBoost ablation using classical graph features (degree, out/in-degree ratio, PageRank, clustering coefficient).
11. A 3-seed robustness check (mean +/- std PR-AUC) on whichever model wins overall.

**Phase 1C (advanced)**
13. A third, differently-aggregating GNN architecture: GIN.
14. GNNExplainer on the best GNN and on GAT at the Phase 1B node, plus a direct edge-ablation check.
15. A camouflage-robustness measurement (CARE-GNN-lite): recall on illicit nodes hiding among non-illicit neighbors.
16. A 50-trial wider search for GraphSAGE (after a search-boundary check) and GAT's first tuning pass.
17. Save every number, every model's weights, and the Phase 1B + Phase 1C closing paragraphs.

**Design decision (see `docs/design_decisions.md`):** this notebook duplicates the minimal logic inline rather than importing from `engines/graph/*.py`, so it runs on Kaggle with nothing uploaded besides this file (only needs internet=on to `pip install` and to let PyG fetch the dataset from its own servers).

**Local vs Kaggle:** set `SMOKE_TEST = True` below to run a tiny CPU-only slice (a few hundred nodes) to catch bugs before spending Kaggle GPU time -- that is the only thing this flag changes. Every real result must come from `SMOKE_TEST = False` on Kaggle with GPU on.


## Kaggle setup (this cell only; no-op if packages are already present)

Accelerator: **GPU**. Internet: **on** (needed both for `pip install` below and for PyG to download the Elliptic dataset from its own servers).

In [ ]:
# Kaggle: accelerator = GPU, internet = ON.
# Installs torch_geometric and xgboost/optuna. Safe to re-run; skips already-satisfied packages quickly.
import sys, subprocess

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    import torch
except ImportError:
    pip_install("torch")
    import torch

TORCH_VER = torch.__version__.split("+")[0]
CUDA_TAG = "cu121" if torch.cuda.is_available() else "cpu"

try:
    import torch_geometric  # noqa: F401
except ImportError:
    pip_install("torch_geometric")

# Neighbor-sampling backend (pyg-lib) is OPTIONAL: the GraphSAGE cell below trains full-batch by
# default (the Elliptic graph is small enough to fit in GPU memory) and only falls back to
# neighbor-sampled mini-batches if that full-batch pass actually OOMs. Best-effort install here;
# a failure must not block the rest of the notebook, since the common path never needs it.
try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyg-lib",
         "-f", f"https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html"],
        check=True,
    )
except subprocess.CalledProcessError:
    print("pyg-lib install failed/unavailable for this torch build -- fine unless the full-batch "
          "GraphSAGE pass OOMs and needs the neighbor-sampling fallback.")

try:
    import xgboost  # noqa: F401
except ImportError:
    pip_install("xgboost")

try:
    import optuna  # noqa: F401
except ImportError:
    pip_install("optuna")

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())


In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.datasets import EllipticBitcoinDataset
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
import xgboost as xgb
import optuna

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Paths: one-line edit between local and Kaggle ---
DATA_ROOT = Path("./data/elliptic")       # Kaggle: Path("/kaggle/working/data/elliptic")
OUTPUT_DIR = Path("./artifacts/graph")    # Kaggle: Path("/kaggle/working/artifacts/graph")
DOCS_DIR = Path("./docs")                 # Kaggle: Path("/kaggle/working/docs")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

# Smoke-test toggle: True = tiny CPU slice to catch bugs, False = the real run.
SMOKE_TEST = False
SMOKE_NODES = 500

DEVICE = torch.device("cuda" if (torch.cuda.is_available() and not SMOKE_TEST) else "cpu")
print("device:", DEVICE, "| SMOKE_TEST:", SMOKE_TEST)


## 1. Data loading + EDA

In [ ]:
dataset = EllipticBitcoinDataset(root=str(DATA_ROOT))
data = dataset[0]

n_nodes = data.num_nodes
n_edges = data.num_edges
n_features = data.num_node_features

print(f"nodes: {n_nodes}")
print(f"edges: {n_edges}")
print(f"features per node: {n_features}")

# Sanity check against the known values from the dataset card / spec.
assert n_nodes == 203769, f"unexpected node count: {n_nodes}"
assert n_features == 165, f"unexpected feature count: {n_features}"
print("node/feature counts match the known Elliptic values.")


In [ ]:
# Labels in PyG's EllipticBitcoinDataset: 0 = licit, 1 = illicit, 2 = unknown.
labels = data.y
label_counts = {int(v): int((labels == v).sum()) for v in labels.unique()}
label_names = {0: "licit", 1: "illicit", 2: "unknown"}
total = n_nodes

print("Label distribution:")
for k in sorted(label_counts):
    c = label_counts[k]
    print(f"  {label_names.get(k, k):8s}: {c:7d}  ({100*c/total:.2f}%)")

# Degree distribution (undirected view for a meaningful degree count).
from torch_geometric.utils import degree, to_undirected
ei_undirected = to_undirected(data.edge_index)
deg = degree(ei_undirected[0], num_nodes=n_nodes)
print(f"\nDegree distribution: mean={deg.mean().item():.3f}  max={int(deg.max().item())}  "
      f"median={deg.median().item():.1f}  nodes with degree 0: {int((deg == 0).sum())}")

# Time-step distribution (49 discrete steps).
# IMPORTANT: PyG's EllipticBitcoinDataset.process() drops both txId and time_step when it builds
# data.x (x = only the 165 real trans_feat_*/agg_feat_* columns), so time_step is NOT a feature
# column -- reading data.x[:, 0] would silently read a normalized feature value instead of a time
# step. The real per-node time step has to come from the raw features CSV that PyG downloaded,
# read in the same row order used to build data.x/data.y (row order is preserved end-to-end).
if hasattr(data, "time_step") and data.time_step is not None:
    time_steps = data.time_step
else:
    import pandas as pd

    features_csv = next(p for p in dataset.raw_paths if "feat" in os.path.basename(p).lower())
    df_features = pd.read_csv(features_csv, header=None)
    time_steps = torch.from_numpy(df_features.iloc[:, 1].to_numpy()).long()
    assert len(time_steps) == n_nodes, (
        f"time_step length {len(time_steps)} != n_nodes {n_nodes} -- row order assumption broke"
    )

uniq, counts = time_steps.unique(return_counts=True)
print(f"\nTime steps present: {len(uniq)} (range {int(uniq.min())}-{int(uniq.max())})")
print(f"nodes per time step: mean={counts.float().mean().item():.1f}  min={int(counts.min())}  max={int(counts.max())}")

# Sanity check: illicit nodes should be spread across many time steps, not clustered near zero --
# this would have caught the data.x[:, 0] bug (it put ~all illicit nodes outside steps 1-30).
illicit_steps = time_steps[labels == 1]
print(f"illicit nodes span time steps {int(illicit_steps.min())}-{int(illicit_steps.max())} "
      f"(mean {illicit_steps.float().mean().item():.1f})")


In [ ]:
label_lines = chr(10).join(
    f"- {label_names.get(k, k)}: {v} ({100*v/total:.2f}%)" for k, v in sorted(label_counts.items())
)

eda_md = f"""# Graph Data Exploration -- Elliptic Bitcoin Dataset

Loaded via `torch_geometric.datasets.EllipticBitcoinDataset`.

## Counts
- Nodes: {n_nodes}
- Edges: {n_edges}
- Features per node: {n_features}

## Label distribution
{label_lines}

Roughly 77% of nodes are unlabeled ("unknown"). These are excluded from train/test metrics
throughout this notebook (semi-supervised setting, not dropped because it was inconvenient).

## Degree distribution
- Mean degree: {deg.mean().item():.3f}
- Median degree: {deg.median().item():.1f}
- Max degree: {int(deg.max().item())}
- Nodes with degree 0: {int((deg == 0).sum())}

Elliptic is known to have a power-law-ish degree distribution with a handful of very
high-degree hub nodes (large mixers/exchanges) sitting far above the median -- visible above
in the mean/median vs. max gap.

## Time steps
- 49 discrete time steps present, range {int(uniq.min())}-{int(uniq.max())}.
- Nodes per step: mean {counts.float().mean().item():.1f}, min {int(counts.min())}, max {int(counts.max())}.
"""

eda_path = DOCS_DIR / "graph_data_exploration.md"
eda_path.write_text(eda_md, encoding="utf-8")
print(f"wrote {eda_path}")


## 2. Temporal split definition

Train: time steps 1-30. Validation buffer: 31-35 (unused in headline metrics, kept for future tuning). Test: 36-49. Unknown-labeled nodes are excluded from every split -- they have no ground truth.

In [ ]:
labeled_mask = labels != 2  # drop "unknown"

train_time_mask = (time_steps >= 1) & (time_steps <= 30)
val_time_mask = (time_steps >= 31) & (time_steps <= 35)
test_time_mask = (time_steps >= 36) & (time_steps <= 49)

train_mask_temporal = labeled_mask & train_time_mask
val_mask_temporal = labeled_mask & val_time_mask
test_mask_temporal = labeled_mask & test_time_mask

print("temporal split (labeled nodes only):")
print(f"  train: {int(train_mask_temporal.sum())}")
print(f"  val:   {int(val_mask_temporal.sum())}")
print(f"  test:  {int(test_mask_temporal.sum())}")

if SMOKE_TEST:
    # Shrink to a tiny slice for a fast local correctness check -- does NOT change the split logic itself.
    def _shrink(mask, n):
        idx = mask.nonzero(as_tuple=True)[0]
        if len(idx) > n:
            keep = idx[torch.randperm(len(idx))[:n]]
            new_mask = torch.zeros_like(mask)
            new_mask[keep] = True
            return new_mask
        return mask
    train_mask_temporal = _shrink(train_mask_temporal, SMOKE_NODES)
    test_mask_temporal = _shrink(test_mask_temporal, SMOKE_NODES // 2)
    print(f"[SMOKE_TEST] shrunk to train={int(train_mask_temporal.sum())} test={int(test_mask_temporal.sum())}")


## 3. Baseline: tuned XGBoost on raw node features

In [ ]:
X = data.x.numpy()
y = labels.numpy()

X_train, y_train = X[train_mask_temporal.numpy()], y[train_mask_temporal.numpy()]
X_test, y_test = X[test_mask_temporal.numpy()], y[test_mask_temporal.numpy()]

pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"train pos (illicit) rate: {(y_train==1).mean():.4f}  |  scale_pos_weight: {pos_weight:.2f}")

N_TRIALS = 5 if SMOKE_TEST else 25

def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "eta": trial.suggest_float("eta", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "scale_pos_weight": pos_weight,
        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "seed": SEED,
    }
    # Inner split of the temporal train set only (never touches the temporal test set).
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train
    )
    model = xgb.XGBClassifier(**params, n_estimators=300, use_label_encoder=False)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    preds = model.predict_proba(X_va)[:, 1]
    return average_precision_score(y_va, preds)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
print("best params:", study.best_params)
print("best inner-val PR-AUC:", study.best_value)


In [ ]:
best_params = {
    **study.best_params,
    "scale_pos_weight": pos_weight,
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "seed": SEED,
}
baseline_model = xgb.XGBClassifier(**best_params, n_estimators=300, use_label_encoder=False)
baseline_model.fit(X_train, y_train)

test_preds = baseline_model.predict_proba(X_test)[:, 1]
pr_auc_baseline_temporal = average_precision_score(y_test, test_preds)
print(f"Baseline (XGBoost, temporal split) PR-AUC: {pr_auc_baseline_temporal:.4f}")


## 4. GraphSAGE model (2 layers) on the same temporal split

In [ ]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=64, out_channels=1):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        return x.squeeze(-1)

data = data.to(DEVICE)
train_mask_temporal_d = train_mask_temporal.to(DEVICE)
test_mask_temporal_d = test_mask_temporal.to(DEVICE)

gnn = GraphSAGE(in_channels=n_features).to(DEVICE)
optimizer = torch.optim.Adam(gnn.parameters(), lr=0.01, weight_decay=5e-4)

# Class-weighted BCE loss, same reasoning as the baseline's scale_pos_weight.
pos_weight_t = torch.tensor([pos_weight], device=DEVICE, dtype=torch.float32)

# Full-batch by default: the Elliptic graph (203K nodes, 234K edges, 165 feats) comfortably fits
# in GPU memory, so there is no reason to pay the extra complexity/dependency cost of neighbor
# sampling unless it turns out not to fit. Only then do we fall back to NeighborLoader mini-batches.
USE_NEIGHBOR_SAMPLING = False
train_loader = None
N_EPOCHS = 5 if SMOKE_TEST else 40


def _make_neighbor_loader():
    return NeighborLoader(
        data,
        num_neighbors=[15, 10],
        batch_size=1024,
        input_nodes=train_mask_temporal_d,
        shuffle=True,
    )


def train_epoch():
    global USE_NEIGHBOR_SAMPLING, train_loader
    gnn.train()

    if not USE_NEIGHBOR_SAMPLING:
        try:
            optimizer.zero_grad()
            out = gnn(data.x, data.edge_index)
            loss = F.binary_cross_entropy_with_logits(
                out[train_mask_temporal_d], data.y[train_mask_temporal_d].float(), pos_weight=pos_weight_t
            )
            loss.backward()
            optimizer.step()
            return loss.item()
        except torch.cuda.OutOfMemoryError:
            print("full-batch forward/backward pass OOM'd on this GPU -- "
                  "falling back to neighbor-sampled mini-batches for the rest of training.")
            torch.cuda.empty_cache()
            USE_NEIGHBOR_SAMPLING = True
            train_loader = _make_neighbor_loader()
            # fall through to the mini-batch loop below for this epoch too

    total_loss = 0.0
    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        out = gnn(batch.x, batch.edge_index)[: batch.batch_size]
        target = batch.y[: batch.batch_size].float()
        loss = F.binary_cross_entropy_with_logits(out, target, pos_weight=pos_weight_t)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
    return total_loss / int(train_mask_temporal_d.sum())


loss_history = []  # per-epoch training loss for the default run -- used by the Phase 1B convergence check
for epoch in range(1, N_EPOCHS + 1):
    loss = train_epoch()
    loss_history.append(loss)
    if epoch % max(1, N_EPOCHS // 5) == 0 or epoch == N_EPOCHS:
        mode = "neighbor-sampled" if USE_NEIGHBOR_SAMPLING else "full-batch"
        print(f"epoch {epoch:3d}  loss {loss:.4f}  ({mode})")


In [ ]:
@torch.no_grad()
def evaluate(mask):
    gnn.eval()
    out = gnn(data.x, data.edge_index)
    probs = torch.sigmoid(out[mask]).cpu().numpy()
    y_true = data.y[mask].cpu().numpy()
    return average_precision_score(y_true, probs)

pr_auc_gnn_temporal = evaluate(test_mask_temporal_d)
print(f"GraphSAGE (temporal split) PR-AUC: {pr_auc_gnn_temporal:.4f}")


## 5. Temporal-shift check: same trained model, random-split evaluation

Same trained GraphSAGE model as above. Instead of the temporal test set, draw a random test set (same overall train/test proportion) from the full pool of labeled nodes, and report PR-AUC there. The gap between this number and the temporal PR-AUC above is the headline finding -- report it plainly whatever it is.

In [ ]:
labeled_idx = labeled_mask.nonzero(as_tuple=True)[0].cpu().numpy()
test_frac = float(test_mask_temporal.sum()) / float(labeled_mask.sum())

if SMOKE_TEST:
    labeled_idx = np.random.RandomState(SEED).choice(labeled_idx, size=min(len(labeled_idx), SMOKE_NODES), replace=False)

_, random_test_idx = train_test_split(
    labeled_idx, test_size=test_frac, random_state=SEED, stratify=labels.numpy()[labeled_idx]
)
random_test_mask = torch.zeros(n_nodes, dtype=torch.bool)
random_test_mask[random_test_idx] = True
random_test_mask = random_test_mask.to(DEVICE)

pr_auc_gnn_random = evaluate(random_test_mask)
gap = pr_auc_gnn_random - pr_auc_gnn_temporal

print(f"GraphSAGE (random split)   PR-AUC: {pr_auc_gnn_random:.4f}")
print(f"GraphSAGE (temporal split) PR-AUC: {pr_auc_gnn_temporal:.4f}")
print(f"Gap (random - temporal):           {gap:+.4f}")


# Phase 1B: Graph Engine Depth

Extends the comparison above per `docs/PHASE_1B_GRAPH_ENGINE_DEPTH.md`: convergence check, a fair tuning pass for GraphSAGE, a GAT model with attention-based explainability, a classical-graph-feature ablation for XGBoost, and a multi-seed robustness check on the overall winner.

## 7. Convergence check on the default GraphSAGE run (Phase 1B, task 1)

Inspect the training loss curve from the default GraphSAGE run above (hidden=64, lr=0.01, dropout=0.3, 40 epochs) that produced the original PR-AUC, and determine whether it was undertrained or had converged.

In [ ]:
import matplotlib.pyplot as plt

print(f"epoch 1 loss: {loss_history[0]:.4f}  |  epoch {len(loss_history)} loss: {loss_history[-1]:.4f}")

# "Still dropping" check: compare the loss drop over the final 20% of training to the loss's
# total range over the whole run. More than a 2% drop in that tail counts as "still dropping".
tail = loss_history[-max(1, len(loss_history) // 5):]
total_range = max(loss_history) - min(loss_history) + 1e-9
tail_drop = tail[0] - tail[-1]
still_dropping = (tail_drop / total_range) > 0.02

convergence_verdict = (
    f"still dropping noticeably in the final epochs -- the default {len(loss_history)}-epoch run was "
    "undertrained; task 2's tuning pass below searches a wider epoch range with early stopping."
    if still_dropping else
    f"flat in the final epochs -- the default run had converged, so its {pr_auc_gnn_temporal:.4f} PR-AUC "
    "reflects a genuinely weaker (not undertrained) GraphSAGE on this temporal split."
)
print(f"convergence verdict: {convergence_verdict}")

plt.figure(figsize=(6, 3.5))
plt.plot(range(1, len(loss_history) + 1), loss_history)
plt.xlabel("epoch")
plt.ylabel("train loss")
plt.title("Default GraphSAGE (hidden=64, lr=0.01) training loss")
plt.tight_layout()
loss_plot_path = OUTPUT_DIR / "graphsage_default_loss_curve.png"
plt.savefig(loss_plot_path, dpi=120)
plt.show()
print(f"wrote {loss_plot_path}")


## 8. Fair tuning pass for GraphSAGE (Phase 1B, task 2)

XGBoost got an Optuna search above; GraphSAGE didn't -- fix that asymmetry. Small Optuna study over hidden dimension, learning rate, dropout, and max epochs, with early stopping on the validation slice (time steps 31-35). Same temporal train/test split as before; if the tuned model still doesn't beat the XGBoost baseline, that is reported as a real result, not a failure to fix.

In [ ]:
val_mask_temporal_d = val_mask_temporal.to(DEVICE)


def train_model_full_batch(model, optimizer, epochs, val_mask=None, patience=None, verbose=False):
    """Generic full-batch training loop reused for GraphSAGE tuning, GAT, and the robustness
    re-runs below. Trains on train_mask_temporal_d; if val_mask is given, tracks the best-val-PR-AUC
    state dict and (optionally) early-stops after `patience` epochs without improvement."""
    loss_history_local = []
    best_val = -1.0
    best_state = None
    epochs_since_improve = 0

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.binary_cross_entropy_with_logits(
            out[train_mask_temporal_d], data.y[train_mask_temporal_d].float(), pos_weight=pos_weight_t
        )
        loss.backward()
        optimizer.step()
        loss_history_local.append(loss.item())

        if val_mask is not None:
            model.eval()
            with torch.no_grad():
                val_out = model(data.x, data.edge_index)
                val_probs = torch.sigmoid(val_out[val_mask]).cpu().numpy()
                val_true = data.y[val_mask].cpu().numpy()
                val_pr_auc = average_precision_score(val_true, val_probs)
            if val_pr_auc > best_val:
                best_val = val_pr_auc
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                epochs_since_improve = 0
            else:
                epochs_since_improve += 1
            if patience is not None and epochs_since_improve >= patience:
                if verbose:
                    print(f"  early stopping at epoch {epoch} (best val PR-AUC {best_val:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return loss_history_local, best_val


@torch.no_grad()
def evaluate_model(model, mask):
    model.eval()
    out = model(data.x, data.edge_index)
    probs = torch.sigmoid(out[mask]).cpu().numpy()
    y_true = data.y[mask].cpu().numpy()
    return average_precision_score(y_true, probs)


In [ ]:
class GraphSAGEFlex(torch.nn.Module):
    """Same architecture as GraphSAGE above, but with hidden_dim/dropout as constructor args
    so Optuna can search over them."""

    def __init__(self, in_channels, hidden_channels=64, dropout=0.3, out_channels=1):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x.squeeze(-1)


N_GNN_TRIALS = 3 if SMOKE_TEST else 15

def gnn_objective(trial):
    hidden_dim = trial.suggest_categorical("hidden_dim", [32, 64, 128])
    lr = trial.suggest_float("lr", 1e-3, 1e-1, log=True)
    dropout = trial.suggest_float("dropout", 0.1, 0.6)
    max_epochs = trial.suggest_int("max_epochs", 20, 100)

    torch.manual_seed(SEED)
    model = GraphSAGEFlex(in_channels=n_features, hidden_channels=hidden_dim, dropout=dropout).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    _, best_val = train_model_full_batch(
        model, optimizer, epochs=max_epochs, val_mask=val_mask_temporal_d, patience=10
    )
    return best_val

gnn_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
gnn_study.optimize(gnn_objective, n_trials=N_GNN_TRIALS, show_progress_bar=False)
print("best GraphSAGE params:", gnn_study.best_params)
print("best val (steps 31-35) PR-AUC during search:", gnn_study.best_value)


In [ ]:
best_gnn_params = gnn_study.best_params
torch.manual_seed(SEED)
best_gnn_model = GraphSAGEFlex(
    in_channels=n_features, hidden_channels=best_gnn_params["hidden_dim"], dropout=best_gnn_params["dropout"]
).to(DEVICE)
best_gnn_optimizer = torch.optim.Adam(best_gnn_model.parameters(), lr=best_gnn_params["lr"], weight_decay=5e-4)

FINAL_MAX_EPOCHS = 20 if SMOKE_TEST else 150
_, _ = train_model_full_batch(
    best_gnn_model, best_gnn_optimizer, epochs=FINAL_MAX_EPOCHS,
    val_mask=val_mask_temporal_d, patience=15, verbose=True,
)

pr_auc_gnn_graphsage_tuned = evaluate_model(best_gnn_model, test_mask_temporal_d)
print(f"GraphSAGE (tuned, temporal split) PR-AUC: {pr_auc_gnn_graphsage_tuned:.4f}")
print(f"  vs. default GraphSAGE (hidden=64, lr=0.01, 40 epochs): {pr_auc_gnn_temporal:.4f}")
print(f"  vs. XGBoost Tier-1 baseline:                           {pr_auc_baseline_temporal:.4f}")


## 9. GAT -- attention-based GNN (Phase 1B, task 3)

Same temporal split, same class-weighting approach as GraphSAGE. GAT's attention weights double as free explainability -- for one correctly-flagged illicit test node, we extract which neighbors the model attended to most (the demo's future "why was this flagged" answer).

In [ ]:
from torch_geometric.nn import GATConv


class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=64, heads=4, dropout=0.3, out_channels=1):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index, return_attention_weights=False):
        if return_attention_weights:
            x, att1 = self.conv1(x, edge_index, return_attention_weights=True)
        else:
            x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        if return_attention_weights:
            x, att2 = self.conv2(x, edge_index, return_attention_weights=True)
            return x.squeeze(-1), att1, att2
        x = self.conv2(x, edge_index)
        return x.squeeze(-1)


torch.manual_seed(SEED)
gat_model = GAT(in_channels=n_features, hidden_channels=64, heads=4, dropout=0.3).to(DEVICE)
gat_optimizer = torch.optim.Adam(gat_model.parameters(), lr=0.01, weight_decay=5e-4)

GAT_EPOCHS = 5 if SMOKE_TEST else 40
gat_loss_history, _ = train_model_full_batch(gat_model, gat_optimizer, epochs=GAT_EPOCHS)

pr_auc_gat_temporal = evaluate_model(gat_model, test_mask_temporal_d)
print(f"GAT (temporal split) PR-AUC: {pr_auc_gat_temporal:.4f}")


In [ ]:
# "Why was this flagged" -- pick a correctly-flagged illicit test node with high confidence and
# report which neighbors GAT's first-layer attention weighted most heavily. This is the free
# explainability signal noted for reuse in the Phase 3 fusion/demo.
gat_model.eval()
with torch.no_grad():
    out, (ei1, alpha1), (ei2, alpha2) = gat_model(data.x, data.edge_index, return_attention_weights=True)
    probs = torch.sigmoid(out).cpu()

alpha1_mean = alpha1.mean(dim=1).cpu()  # average attention across heads
src, dst = ei1.cpu()

# Some nodes have no *incoming* edges in the directed graph (they only send payments), so under
# directed message passing they receive nothing but a GATConv-added self-loop -- a useless
# explainability example even if the model is very confident. (They are not isolated: the EDA
# shows no degree-0 nodes.) So only consider nodes with >=1 real incoming edge.
real_neighbor_counts = torch.zeros(n_nodes, dtype=torch.long)
real_neighbor_counts.scatter_add_(0, dst[src != dst], torch.ones((src != dst).sum(), dtype=torch.long))
has_real_neighbors = real_neighbor_counts > 0

test_idx = test_mask_temporal_d.nonzero(as_tuple=True)[0].cpu()
y_true_test = data.y[test_idx].cpu()
probs_test = probs[test_idx]
eligible = test_idx[has_real_neighbors[test_idx]]
eligible_y = data.y[eligible].cpu()
eligible_probs = probs[eligible]

correct_illicit = eligible[(eligible_y == 1) & (eligible_probs > 0.5)]

if len(correct_illicit) == 0:
    print("no confidently-correct (p>0.5) illicit test node with real neighbors found -- "
          "falling back to the highest-scored illicit test node that has real neighbors")
    illicit_test = eligible[eligible_y == 1]
    target_node = illicit_test[probs[illicit_test].argmax()].item()
else:
    target_node = correct_illicit[probs[correct_illicit].argmax()].item()

incoming = ((dst == target_node) & (src != target_node)).nonzero(as_tuple=True)[0]
neighbor_scores = sorted(
    ((src[i].item(), alpha1_mean[i].item()) for i in incoming),
    key=lambda t: t[1], reverse=True,
)

print(f"target illicit node: {target_node}  (predicted prob {probs[target_node].item():.4f}, "
      f"{len(neighbor_scores)} real neighbors)")
print("top attended neighbors (node_id, attention_weight):")
for neighbor, weight in neighbor_scores[:5]:
    print(f"  {neighbor}: {weight:.4f}")

gat_explanation = {
    "target_node": int(target_node),
    "predicted_prob": float(probs[target_node].item()),
    "top_attended_neighbors": [{"node": int(n), "attention_weight": float(w)} for n, w in neighbor_scores[:5]],
}


## 10. Tier-2 graph-feature ablation for XGBoost (Phase 1B, task 4)

Compute classical graph features per node (degree, out/in-degree ratio, PageRank, clustering coefficient via NetworkX) and retrain XGBoost on Tier-1 (raw features) + Tier-2 (graph features) combined, same temporal split. Compared against the Tier-1-only baseline and the GNNs above -- whichever way this lands is a real, reportable finding.

In [ ]:
try:
    import networkx as nx
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "networkx"], check=True)
    import networkx as nx

print("building NetworkX graphs for Tier-2 feature computation (this may take a minute)...")

edge_list_undirected = ei_undirected.cpu().t().numpy().tolist()
edge_list_directed = data.edge_index.cpu().t().numpy().tolist()

G_undirected = nx.Graph()
G_undirected.add_nodes_from(range(n_nodes))
G_undirected.add_edges_from(edge_list_undirected)

G_directed = nx.DiGraph()
G_directed.add_nodes_from(range(n_nodes))
G_directed.add_edges_from(edge_list_directed)

degree_arr = deg.cpu().numpy()  # already computed in the EDA cell (undirected degree)

in_deg = dict(G_directed.in_degree())
out_deg = dict(G_directed.out_degree())
in_deg_arr = np.array([in_deg[i] for i in range(n_nodes)], dtype=np.float32)
out_deg_arr = np.array([out_deg[i] for i in range(n_nodes)], dtype=np.float32)
io_ratio_arr = (out_deg_arr + 1.0) / (in_deg_arr + 1.0)  # +1 smoothing to avoid div-by-zero

print("computing PageRank...")
pagerank = nx.pagerank(G_directed, tol=1e-4, max_iter=100)
pagerank_arr = np.array([pagerank[i] for i in range(n_nodes)], dtype=np.float32)

print("computing clustering coefficient...")
clustering = nx.clustering(G_undirected)
clustering_arr = np.array([clustering[i] for i in range(n_nodes)], dtype=np.float32)

tier2_features = np.stack([degree_arr, io_ratio_arr, pagerank_arr, clustering_arr], axis=1)
print(f"Tier-2 feature matrix: {tier2_features.shape} (degree, out/in-degree ratio, pagerank, clustering)")

X_tier12 = np.concatenate([X, tier2_features], axis=1)
X_tier12_train = X_tier12[train_mask_temporal.numpy()]
X_tier12_test = X_tier12[test_mask_temporal.numpy()]
print(f"X_tier12_train: {X_tier12_train.shape}  X_tier12_test: {X_tier12_test.shape}")


In [ ]:
N_TIER2_TRIALS = 5 if SMOKE_TEST else 25

def tier2_objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "eta": trial.suggest_float("eta", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "scale_pos_weight": pos_weight,
        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "seed": SEED,
    }
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_tier12_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train
    )
    model = xgb.XGBClassifier(**params, n_estimators=300, use_label_encoder=False)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    preds = model.predict_proba(X_va)[:, 1]
    return average_precision_score(y_va, preds)

tier2_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
tier2_study.optimize(tier2_objective, n_trials=N_TIER2_TRIALS, show_progress_bar=False)

tier2_best_params = {
    **tier2_study.best_params,
    "scale_pos_weight": pos_weight,
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "seed": SEED,
}
tier12_model = xgb.XGBClassifier(**tier2_best_params, n_estimators=300, use_label_encoder=False)
tier12_model.fit(X_tier12_train, y_train)
tier12_preds = tier12_model.predict_proba(X_tier12_test)[:, 1]
pr_auc_baseline_tier12_temporal = average_precision_score(y_test, tier12_preds)

print(f"XGBoost Tier-1 only PR-AUC:        {pr_auc_baseline_temporal:.4f}")
print(f"XGBoost Tier-1 + Tier-2 PR-AUC:    {pr_auc_baseline_tier12_temporal:.4f}")
print(f"Tuned GraphSAGE PR-AUC:            {pr_auc_gnn_graphsage_tuned:.4f}")
print(f"GAT PR-AUC:                        {pr_auc_gat_temporal:.4f}")


## 11. Robustness check across seeds (Phase 1B, task 5)

Re-run whichever model wins overall (tuned GraphSAGE / GAT / XGBoost Tier-1+Tier-2) across 3 random seeds on the same temporal split, and report mean ± std PR-AUC instead of a single number.

In [ ]:
candidates = {
    "graphsage_tuned": pr_auc_gnn_graphsage_tuned,
    "gat": pr_auc_gat_temporal,
    "xgboost_tier1_tier2": pr_auc_baseline_tier12_temporal,
}
best_model_name = max(candidates, key=candidates.get)
print(f"best-performing model overall: {best_model_name} (PR-AUC {candidates[best_model_name]:.4f})")

SEEDS_FOR_ROBUSTNESS = [SEED, SEED + 1, SEED + 2]
robustness_scores = []

for s in SEEDS_FOR_ROBUSTNESS:
    if best_model_name == "graphsage_tuned":
        torch.manual_seed(s)
        m = GraphSAGEFlex(
            in_channels=n_features, hidden_channels=best_gnn_params["hidden_dim"], dropout=best_gnn_params["dropout"]
        ).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=best_gnn_params["lr"], weight_decay=5e-4)
        train_model_full_batch(m, opt, epochs=FINAL_MAX_EPOCHS, val_mask=val_mask_temporal_d, patience=15)
        score = evaluate_model(m, test_mask_temporal_d)
    elif best_model_name == "gat":
        torch.manual_seed(s)
        m = GAT(in_channels=n_features, hidden_channels=64, heads=4, dropout=0.3).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=0.01, weight_decay=5e-4)
        train_model_full_batch(m, opt, epochs=GAT_EPOCHS)
        score = evaluate_model(m, test_mask_temporal_d)
    else:  # xgboost_tier1_tier2
        params_s = {**tier2_best_params, "seed": s}
        m = xgb.XGBClassifier(**params_s, n_estimators=300, use_label_encoder=False)
        m.fit(X_tier12_train, y_train)
        preds = m.predict_proba(X_tier12_test)[:, 1]
        score = average_precision_score(y_test, preds)
    robustness_scores.append(float(score))
    print(f"  seed {s}: PR-AUC {score:.4f}")

robustness_mean = float(np.mean(robustness_scores))
robustness_std = float(np.std(robustness_scores))
print(f"\n{best_model_name}: {robustness_mean:.4f} +/- {robustness_std:.4f} PR-AUC across "
      f"{len(SEEDS_FOR_ROBUSTNESS)} seeds -- this is the defensible number, not the single-seed one above.")


# Phase 1C: Graph Engine Advanced

Extends Phase 1 + 1B per `docs/PHASE_1C_GRAPH_ENGINE_ADVANCED.md`: a third GNN architecture, GNNExplainer-based explainability, a camouflage-robustness measurement, and a bigger hyperparameter search. Each task saves its result to `artifacts/graph/results_phase1c_partial.json` as soon as it finishes, so a crash or timeout part-way still leaves a record of what was done.

## 13. Third GNN architecture: GIN (Phase 1C, task 1)

RGCN needs typed edges; Elliptic's edges are untyped, so RGCN would collapse to a single-relation GCN, and plain GCN (normalized mean aggregation) is too close to GraphSAGE to count as a different family. **GIN** (sum aggregation + MLP) is used instead: a genuinely different aggregator from GraphSAGE's mean and GAT's attention-weighted mean, and provably more expressive at distinguishing neighborhoods. Same temporal split, class-weighted loss, defaults, and PR-AUC metric as GAT in Phase 1B.

In [ ]:
from torch_geometric.nn import GINConv

# Phase 1C tasks are independent. Each one writes its result here as soon as it finishes, so if a
# later task crashes or the Kaggle session runs out, there's still a record of what completed.
PHASE1C_PARTIAL_PATH = OUTPUT_DIR / "results_phase1c_partial.json"
PHASE1C_PARTIAL_PATH.write_text("{}", encoding="utf-8")  # fresh for this run


def save_phase1c_progress(key, value):
    partial = json.loads(PHASE1C_PARTIAL_PATH.read_text(encoding="utf-8"))
    partial[key] = value
    PHASE1C_PARTIAL_PATH.write_text(json.dumps(partial, indent=2), encoding="utf-8")
    print(f"[phase 1c progress] saved '{key}' -> {PHASE1C_PARTIAL_PATH}")


class GIN(torch.nn.Module):
    """Graph Isomorphism Network: *sum* aggregation over neighbors followed by an MLP. A different
    aggregation family from GraphSAGE (mean) and GAT (attention-weighted mean) -- sum+MLP is
    injective over neighbor multisets, so it can tell apart neighborhoods that mean-style
    aggregators collapse together (e.g. 1 illicit neighbor vs. 10).

    BatchNorm inside each MLP follows the original GIN paper; without it, summing over Elliptic's
    hub nodes (degree up to 473) can blow up activations."""

    def __init__(self, in_channels, hidden_channels=64, dropout=0.3, out_channels=1):
        super().__init__()
        self.conv1 = GINConv(torch.nn.Sequential(
            torch.nn.Linear(in_channels, hidden_channels), torch.nn.BatchNorm1d(hidden_channels), torch.nn.ReLU(),
            torch.nn.Linear(hidden_channels, hidden_channels),
        ))
        self.conv2 = GINConv(torch.nn.Sequential(
            torch.nn.Linear(hidden_channels, hidden_channels), torch.nn.BatchNorm1d(hidden_channels), torch.nn.ReLU(),
            torch.nn.Linear(hidden_channels, out_channels),
        ))
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x.squeeze(-1)


# Same defaults and training loop GAT got in Phase 1B (hidden=64, dropout=0.3, lr=0.01, 40 epochs,
# class-weighted BCE), so the three architectures are compared on equal footing before any tuning.
torch.manual_seed(SEED)
gin_model = GIN(in_channels=n_features, hidden_channels=64, dropout=0.3).to(DEVICE)
gin_optimizer = torch.optim.Adam(gin_model.parameters(), lr=0.01, weight_decay=5e-4)

GIN_EPOCHS = 5 if SMOKE_TEST else 40
gin_loss_history, _ = train_model_full_batch(gin_model, gin_optimizer, epochs=GIN_EPOCHS)
if not np.isfinite(gin_loss_history[-1]):
    raise RuntimeError(f"GIN training diverged (final loss {gin_loss_history[-1]}) -- log this as a real bug before continuing")
pr_auc_gin_temporal = evaluate_model(gin_model, test_mask_temporal_d)

print(f"GIN (temporal split) PR-AUC:   {pr_auc_gin_temporal:.4f}")
print(f"  vs. GraphSAGE (tuned):       {pr_auc_gnn_graphsage_tuned:.4f}")
print(f"  vs. GAT:                     {pr_auc_gat_temporal:.4f}")
print(f"  vs. XGBoost Tier-1 baseline: {pr_auc_baseline_temporal:.4f}")
print(f"  final train loss: {gin_loss_history[-1]:.4f} (epoch 1: {gin_loss_history[0]:.4f})")

save_phase1c_progress("gin", {
    "pr_auc_temporal": float(pr_auc_gin_temporal),
    "config": {"hidden_dim": 64, "dropout": 0.3, "lr": 0.01, "epochs": GIN_EPOCHS, "batchnorm_in_mlps": True},
    "train_loss_first_last": [float(gin_loss_history[0]), float(gin_loss_history[-1])],
})


## 14. GNNExplainer for proper explainability (Phase 1C, task 2)

Run PyG's `Explainer` + `GNNExplainer` on the best GNN so far, at the Phase 1B target node if that model also flags it correctly (otherwise its own best correctly-flagged illicit node). Also run it on GAT at the exact node whose raw attention weight was reported in Phase 1B, so the attention-vs-GNNExplainer comparison uses the same model. Each explanation also gets a direct edge-ablation check (remove every edge touching the node, measure the prediction change), and reports whether the important features sit in Elliptic's local block (0-92) or its precomputed neighborhood-aggregate block (93-164).

In [ ]:
from torch_geometric.explain import Explainer, GNNExplainer

# Elliptic's 165 features: indices 0-92 are local transaction features; 93-164 are one-hop
# neighborhood aggregates precomputed by the dataset authors. Which block GNNExplainer leans on says
# whether "graph information" is reaching the model through message passing or was already baked
# into the raw features.
N_LOCAL_FEATURES = 93
LOCAL_COUNT_SHARE = N_LOCAL_FEATURES / n_features  # a uniform mask would give exactly this share

# Best GNN so far (before task 4's wider search) -- also used by the camouflage check below.
gnn_candidates = {
    "graphsage_tuned": (best_gnn_model, pr_auc_gnn_graphsage_tuned),
    "gat": (gat_model, pr_auc_gat_temporal),
    "gin": (gin_model, pr_auc_gin_temporal),
}
best_gnn_for_explain_name = max(gnn_candidates, key=lambda k: gnn_candidates[k][1])
best_gnn_for_explain = gnn_candidates[best_gnn_for_explain_name][0]
best_gnn_for_camo_name, best_gnn_for_camo = best_gnn_for_explain_name, best_gnn_for_explain
print(f"best GNN so far: {best_gnn_for_explain_name} (PR-AUC {gnn_candidates[best_gnn_for_explain_name][1]:.4f})")


@torch.no_grad()
def _node_probs(model, edge_index):
    model.eval()
    return torch.sigmoid(model(data.x, edge_index)).cpu()


def pick_target(model):
    """Reuse the Phase 1B target node if this model also flags it correctly; otherwise take this
    model's most confident correctly-flagged illicit test node that has real neighbors."""
    p = _node_probs(model, data.edge_index)
    if p[target_node] > 0.5:
        return target_node, "reused Phase 1B target node (also correctly flagged by this model)"
    cand = eligible[(eligible_y == 1) & (p[eligible] > 0.5)]
    if len(cand) > 0:
        return cand[p[cand].argmax()].item(), "Phase 1B target not flagged by this model; used its best correctly-flagged illicit node"
    cand = eligible[eligible_y == 1]
    return cand[p[cand].argmax()].item(), "no correctly-flagged illicit node with neighbors; used the highest-scored one"


def explain_node(model, model_name, node, epochs=200):
    explainer = Explainer(
        model=model,
        algorithm=GNNExplainer(epochs=epochs),
        explanation_type="model",
        node_mask_type="attributes",
        edge_mask_type="object",
        model_config=dict(mode="binary_classification", task_level="node", return_type="raw"),
    )
    torch.manual_seed(SEED)
    expl = explainer(data.x, data.edge_index, index=node)

    feat_imp = expl.node_mask[node].detach().cpu()
    top_feat = torch.argsort(feat_imp, descending=True)[:10]
    local_mass = float(feat_imp[:N_LOCAL_FEATURES].sum())
    agg_mass = float(feat_imp[N_LOCAL_FEATURES:].sum())
    total_mass = max(local_mass + agg_mass, 1e-12)
    # Most features sit near a common mask floor; only features well above it carry signal.
    floor = float(feat_imp.median())
    standout = [int(i) for i in top_feat if feat_imp[i] > 2 * floor]

    edge_mask = expl.edge_mask.detach().cpu()
    ei = data.edge_index.cpu()
    incident = ((ei[0] == node) | (ei[1] == node)).nonzero(as_tuple=True)[0]
    incident = incident[torch.argsort(edge_mask[incident], descending=True)]
    in_subgraph = (edge_mask > 0).nonzero(as_tuple=True)[0]  # edges in the node's computation subgraph
    top_edges = in_subgraph[torch.argsort(edge_mask[in_subgraph], descending=True)][:5]

    # Direct fidelity check: remove every edge touching the node. For a 2-layer GNN every message
    # path into the node ends on one of these edges, so this cuts all neighborhood information.
    keep = ((ei[0] != node) & (ei[1] != node)).to(DEVICE)
    p_full = _node_probs(model, data.edge_index)[node].item()
    p_cut = _node_probs(model, data.edge_index[:, keep])[node].item()

    result = {
        "model": model_name,
        "target_node": int(node),
        "prob_full_graph": p_full,
        "prob_all_incident_edges_removed": p_cut,
        "prob_change_from_edge_removal": p_cut - p_full,
        "top_features": [
            {"feature_index": int(i), "block": "local" if i < N_LOCAL_FEATURES else "neighborhood_aggregate",
             "importance": float(feat_imp[i])}
            for i in top_feat
        ],
        "feature_mask_floor_median": floor,
        "standout_features": [
            {"feature_index": i, "block": "local" if i < N_LOCAL_FEATURES else "neighborhood_aggregate",
             "importance": float(feat_imp[i])}
            for i in standout
        ],
        "feature_mask_share_local": local_mass / total_mass,
        "feature_mask_share_local_if_uniform": LOCAL_COUNT_SHARE,
        "feature_mask_share_neighborhood_aggregate": agg_mass / total_mass,
        "incident_edges": [
            {"src": int(ei[0, i]), "dst": int(ei[1, i]), "importance": float(edge_mask[i])} for i in incident[:5]
        ],
        "top_edges_in_computation_subgraph": [
            {"src": int(ei[0, i]), "dst": int(ei[1, i]), "importance": float(edge_mask[i])} for i in top_edges
        ],
        "computation_subgraph_edges": int(len(in_subgraph)),
    }

    print(f"\n=== GNNExplainer: {model_name}, node {node} ===")
    print(f"prob full graph {p_full:.4f} | all incident edges removed {p_cut:.4f} (change {p_cut - p_full:+.4f})")
    print(f"feature-mask share: local {result['feature_mask_share_local']:.1%} "
          f"(a uniform mask would give {LOCAL_COUNT_SHARE:.1%}) | "
          f"neighborhood-aggregate {result['feature_mask_share_neighborhood_aggregate']:.1%}")
    print(f"mask floor (median) {floor:.3f}; standout features (>2x floor):",
          [(d["feature_index"], d["block"][:3], round(d["importance"], 3)) for d in result["standout_features"]])
    print("top-10 features:", ", ".join(f"f{d['feature_index']}({d['block'][:3]}):{d['importance']:.3f}"
                                       for d in result["top_features"]))
    print(f"incident edges ({len(incident)}):", result["incident_edges"])
    print(f"top edges in computation subgraph ({result['computation_subgraph_edges']} total):",
          result["top_edges_in_computation_subgraph"])
    return result


explain_target, explain_target_reason = pick_target(best_gnn_for_explain)
print(f"target for {best_gnn_for_explain_name}: node {explain_target} -- {explain_target_reason}")
gnn_explainer_best = explain_node(best_gnn_for_explain, best_gnn_for_explain_name, explain_target)
gnn_explainer_best["target_selection"] = explain_target_reason

# Same model, two explanation methods: run GNNExplainer on GAT at the exact node whose raw attention
# weight was reported in Phase 1B, so the attention-vs-GNNExplainer comparison is apples-to-apples.
gnn_explainer_gat = (
    gnn_explainer_best if best_gnn_for_explain_name == "gat"
    else explain_node(gat_model, "gat", target_node)
)


def describe_explanation(r):
    """Three separate signals, stated separately -- they don't have to agree."""
    delta = r["prob_change_from_edge_removal"]
    max_edge = max((e["importance"] for e in r["incident_edges"]), default=0.0)
    neighbors_matter = abs(delta) > 0.05
    ablation = (f"cutting all of node {r['target_node']}'s edges changes {r['model']}'s prediction by {delta:+.4f}, "
                + ("so neighborhood messages do matter" if neighbors_matter else "so neighborhood messages don't matter"))
    if (max_edge > 0.5) == neighbors_matter:
        mask = f"GNNExplainer's edge mask (max {max_edge:.3f}) is consistent with that"
    else:
        mask = (f"GNNExplainer's edge mask (max {max_edge:.3f}) contradicts that, so on this node the mask isn't "
                "faithful to the model and needs an ablation check")
    shift = r["feature_mask_share_local"] - r["feature_mask_share_local_if_uniform"]
    if r["standout_features"]:
        blocks = sorted({d["block"] for d in r["standout_features"]})
        feats = (f"standout features {[d['feature_index'] for d in r['standout_features']]} are in "
                 f"{' and '.join(blocks)} (overall local share {r['feature_mask_share_local']:.0%} vs "
                 f"{r['feature_mask_share_local_if_uniform']:.0%} for a uniform mask, shift {shift:+.0%})")
    else:
        feats = (f"no feature stands out above the mask floor (local share {r['feature_mask_share_local']:.0%} vs "
                 f"{r['feature_mask_share_local_if_uniform']:.0%} for a uniform mask)")
    return f"{ablation}; {mask}; {feats}."


gat_attn_weight = gat_explanation["top_attended_neighbors"][0]["attention_weight"] if gat_explanation["top_attended_neighbors"] else None
attention_says_neighbor_matters = gat_attn_weight is not None and gat_attn_weight > 0.1
ablation_says_neighbor_matters = abs(gnn_explainer_gat["prob_change_from_edge_removal"]) > 0.05
agreement = "agrees" if attention_says_neighbor_matters == ablation_says_neighbor_matters else "disagrees"

gnn_explainer_vs_attention_finding = (
    f"On GAT at node {target_node} the Phase 1B attention weight was {gat_attn_weight:.2e}, and the edge-ablation "
    f"check {agreement} with it: {describe_explanation(gnn_explainer_gat)}"
)
best_gnn_explanation_summary = describe_explanation(gnn_explainer_best)
print(f"\n{gnn_explainer_vs_attention_finding}")
print(f"\nBest GNN: {best_gnn_explanation_summary}")

gnn_explainer_result = {
    "best_gnn": gnn_explainer_best,
    "gat_same_node_as_phase1b": gnn_explainer_gat,
    "phase1b_gat_attention_weight": gat_attn_weight,
    "comparison_to_attention": gnn_explainer_vs_attention_finding,
    "best_gnn_summary": best_gnn_explanation_summary,
}
save_phase1c_progress("gnn_explainer", gnn_explainer_result)


## 15. Camouflage-robustness check, CARE-GNN-lite (Phase 1C, task 3)

Measure (not defend against) camouflage: illicit test nodes whose real neighbors are mostly or entirely non-illicit. Two definitions -- "mostly licit/unknown" (<50% illicit neighbors) and strict "zero illicit neighbors" -- each compared against the non-camouflaged illicit nodes that have neighbors. The verdict is based on recall (prevalence-independent); PR-AUC is shown next to its chance baseline because it isn't comparable across subsets of different sizes.

In [ ]:
# Neighbor label composition over the undirected graph. Labels of neighbors are used only to
# *define evaluation subsets* here -- nothing below is fed back into training.
src_u, dst_u = ei_undirected.cpu()
not_self = src_u != dst_u
is_illicit_global = (labels == 1).cpu()

real_degree_u = torch.zeros(n_nodes, dtype=torch.long)
real_degree_u.scatter_add_(0, dst_u[not_self], torch.ones(int(not_self.sum()), dtype=torch.long))
illicit_neighbor_count = torch.zeros(n_nodes, dtype=torch.long)
illicit_neighbor_count.scatter_add_(0, dst_u[not_self], is_illicit_global[src_u[not_self]].long())
illicit_neighbor_frac = torch.where(
    real_degree_u > 0, illicit_neighbor_count.float() / real_degree_u.clamp(min=1).float(), torch.zeros(n_nodes)
)

test_illicit_idx = (test_mask_temporal & (labels == 1)).nonzero(as_tuple=True)[0].cpu()
licit_test_idx = (test_mask_temporal & (labels == 0)).nonzero(as_tuple=True)[0].cpu()

# Two definitions, both restricted to illicit nodes that actually have neighbors (an isolated node
# isn't "hiding among" anything):
#   "mostly"  -- under half of its real neighbors are labeled illicit (the spec's wording; note that
#                with ~77% of all nodes unlabeled, most neighborhoods are dominated by unknowns)
#   "strict"  -- zero of its real neighbors are labeled illicit
CAMOUFLAGE_DEFINITIONS = {
    "mostly_licit_or_unknown": (real_degree_u > 0) & (illicit_neighbor_frac < 0.5),
    "zero_illicit_neighbors": (real_degree_u > 0) & (illicit_neighbor_count == 0),
}
MIN_SUBSET_SIZE = 20  # below this, recall on the subset is too noisy to report as a finding

best_gnn_for_camo.eval()
with torch.no_grad():
    camo_probs = torch.sigmoid(best_gnn_for_camo(data.x, data.edge_index)).cpu()


def _subset_metrics(pos_idx):
    """Recall at p>0.5 (prevalence-independent -- this is what the verdict uses), plus PR-AUC against
    the shared licit-test negative pool reported *next to its chance-level baseline*, since PR-AUC
    shifts with how many positives a subset has and isn't directly comparable across subset sizes."""
    if len(pos_idx) == 0:
        return {"n": 0, "recall": None, "pr_auc_vs_licit": None, "pr_auc_chance_baseline": None}
    idx = torch.cat([pos_idx, licit_test_idx])
    return {
        "n": int(len(pos_idx)),
        "recall": float((camo_probs[pos_idx] > 0.5).float().mean()),
        "pr_auc_vs_licit": float(average_precision_score(data.y[idx].cpu().numpy(), camo_probs[idx].numpy())),
        "pr_auc_chance_baseline": float(len(pos_idx) / len(idx)),
    }


def _fmt(m):
    if m["n"] == 0:
        return "n=0"
    return (f"n={m['n']:5d}  recall={m['recall']:.4f}  "
            f"PR-AUC={m['pr_auc_vs_licit']:.4f} (chance {m['pr_auc_chance_baseline']:.4f})")


camouflage_results = {
    "model": best_gnn_for_camo_name,
    "illicit_test_total": int(len(test_illicit_idx)),
    "isolated_illicit_test_nodes": int((real_degree_u[test_illicit_idx] == 0).sum()),
    "all_illicit": _subset_metrics(test_illicit_idx),
    "definitions": {},
}
print(f"model: {best_gnn_for_camo_name}  |  illicit test nodes: {len(test_illicit_idx)} "
      f"(isolated: {camouflage_results['isolated_illicit_test_nodes']})")
print(f"  all illicit       {_fmt(camouflage_results['all_illicit'])}")

camouflage_findings = []
for def_name, camo_mask in CAMOUFLAGE_DEFINITIONS.items():
    camo_idx = test_illicit_idx[camo_mask[test_illicit_idx]]
    # "non-camouflaged" = has neighbors but doesn't meet the definition (isolated nodes excluded from both)
    non_camo_idx = test_illicit_idx[(~camo_mask[test_illicit_idx]) & (real_degree_u[test_illicit_idx] > 0)]
    c, nc = _subset_metrics(camo_idx), _subset_metrics(non_camo_idx)
    entry = {"camouflaged": c, "non_camouflaged": nc}
    camouflage_results["definitions"][def_name] = entry

    print(f"\n[{def_name}]")
    print(f"  camouflaged       {_fmt(c)}")
    print(f"  non-camouflaged   {_fmt(nc)}")

    if min(c["n"], nc["n"]) < MIN_SUBSET_SIZE:
        finding = (f"[{def_name}] too few nodes on one side (camouflaged n={c['n']}, non-camouflaged n={nc['n']}) "
                   f"to compare reliably -- not reported as a finding.")
    else:
        recall_gap = nc["recall"] - c["recall"]
        if recall_gap > 0.10:
            finding = (f"[{def_name}] recall drops sharply on camouflaged illicit nodes ({c['recall']:.4f} vs "
                       f"{nc['recall']:.4f} non-camouflaged, gap {recall_gap:+.4f}) -- {best_gnn_for_camo_name} is "
                       f"measurably fooled by fraud hiding among non-illicit neighbors.")
        elif recall_gap < -0.10:
            finding = (f"[{def_name}] recall is actually *higher* on camouflaged illicit nodes ({c['recall']:.4f} vs "
                       f"{nc['recall']:.4f} non-camouflaged, gap {recall_gap:+.4f}) -- the opposite of the camouflage "
                       f"vulnerability the plan predicted.")
        else:
            direction = ("slightly lower" if recall_gap > 0 else "slightly higher") if abs(recall_gap) > 0.02 else "about the same"
            finding = (f"[{def_name}] no sharp recall drop on camouflaged illicit nodes ({c['recall']:.4f} vs "
                       f"{nc['recall']:.4f} non-camouflaged, gap {recall_gap:+.4f}; camouflaged recall is {direction}) -- "
                       f"on this data {best_gnn_for_camo_name} is not meaningfully fooled by 1-hop camouflage.")
    entry["finding"] = finding
    camouflage_findings.append(finding)
    print(f"  -> {finding}")

camouflage_finding = " ".join(camouflage_findings)
label_coverage_note = (
    f"Caveat: {float((labels == 2).float().mean()):.0%} of nodes are unlabeled, so 'no illicit neighbors' "
    "mostly means 'unlabeled neighbors' -- this measures camouflage relative to known labels, not true hidden fraud."
)
print(label_coverage_note)
camouflage_results["label_coverage_note"] = label_coverage_note
save_phase1c_progress("camouflage_check", camouflage_results)


## 16. Bigger hyperparameter search (Phase 1C, task 4)

Re-run the tuning with 50 trials. For GraphSAGE, first check whether the Phase 1B best `hidden_dim` sat at the edge of its search space (widen upward if so). GAT never got an Optuna search in Phase 1B, so this is its first real tuning pass, compared against its untuned default. A plateau is reported as a plateau -- more trials don't automatically help.

In [ ]:
# Check first: was the Phase 1B best hidden_dim at the edge of its search space? If so, widen the
# range upward; otherwise keep the same space and only add trials.
PREV_HIDDEN_OPTIONS = [32, 64, 128]  # the Phase 1B GraphSAGE search space
was_at_boundary = best_gnn_params["hidden_dim"] == max(PREV_HIDDEN_OPTIONS)
WIDE_HIDDEN_OPTIONS = [64, 128, 256] if was_at_boundary else PREV_HIDDEN_OPTIONS
print(f"Phase 1B best GraphSAGE hidden_dim: {best_gnn_params['hidden_dim']} "
      f"(search-space max was {max(PREV_HIDDEN_OPTIONS)}) -> "
      + (f"at the boundary, widening hidden_dim to {WIDE_HIDDEN_OPTIONS}"
         if was_at_boundary else "not at the boundary, same space, more trials"))

WIDE_N_TRIALS = 5 if SMOKE_TEST else 50

def gnn_objective_wide(trial):
    hidden_dim = trial.suggest_categorical("hidden_dim", WIDE_HIDDEN_OPTIONS)
    lr = trial.suggest_float("lr", 1e-3, 1e-1, log=True)
    dropout = trial.suggest_float("dropout", 0.1, 0.6)
    max_epochs = trial.suggest_int("max_epochs", 20, 100)

    torch.manual_seed(SEED)
    model = GraphSAGEFlex(in_channels=n_features, hidden_channels=hidden_dim, dropout=dropout).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    _, best_val = train_model_full_batch(
        model, optimizer, epochs=max_epochs, val_mask=val_mask_temporal_d, patience=10
    )
    return best_val

gnn_study_wide = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
gnn_study_wide.optimize(gnn_objective_wide, n_trials=WIDE_N_TRIALS, show_progress_bar=False)
print("wider-search best GraphSAGE params:", gnn_study_wide.best_params)
print("best val (steps 31-35) PR-AUC during wider search:", gnn_study_wide.best_value)


In [ ]:
best_gnn_params_wide = gnn_study_wide.best_params
torch.manual_seed(SEED)
best_gnn_model_wide = GraphSAGEFlex(
    in_channels=n_features, hidden_channels=best_gnn_params_wide["hidden_dim"], dropout=best_gnn_params_wide["dropout"]
).to(DEVICE)
best_gnn_optimizer_wide = torch.optim.Adam(best_gnn_model_wide.parameters(), lr=best_gnn_params_wide["lr"], weight_decay=5e-4)
train_model_full_batch(
    best_gnn_model_wide, best_gnn_optimizer_wide, epochs=FINAL_MAX_EPOCHS,
    val_mask=val_mask_temporal_d, patience=15, verbose=True,
)
pr_auc_gnn_graphsage_wide_tuned = evaluate_model(best_gnn_model_wide, test_mask_temporal_d)

graphsage_wide_change = pr_auc_gnn_graphsage_wide_tuned - pr_auc_gnn_graphsage_tuned
graphsage_wide_verdict = (
    f"meaningfully changed ({graphsage_wide_change:+.4f})" if abs(graphsage_wide_change) > 0.01
    else f"plateaued ({graphsage_wide_change:+.4f}) -- the Phase 1B search was already good enough"
)
wide_at_boundary_again = [
    name for name, hit in [
        ("hidden_dim", best_gnn_params_wide["hidden_dim"] == max(WIDE_HIDDEN_OPTIONS)),
        ("max_epochs", best_gnn_params_wide["max_epochs"] == 100),
    ] if hit
]
if wide_at_boundary_again:
    graphsage_wide_verdict += f"; best config is at the widened boundary again ({', '.join(wide_at_boundary_again)}), so not a plateau"
print(f"GraphSAGE wider-search PR-AUC: {pr_auc_gnn_graphsage_wide_tuned:.4f} "
      f"(vs. Phase 1B tuned: {pr_auc_gnn_graphsage_tuned:.4f}) -- {graphsage_wide_verdict}")

save_phase1c_progress("wider_search_graphsage", {
    "phase1b_best_hidden_dim": int(best_gnn_params["hidden_dim"]),
    "was_at_boundary": bool(was_at_boundary),
    "hidden_dim_options": WIDE_HIDDEN_OPTIONS,
    "n_trials": WIDE_N_TRIALS,
    "best_params": best_gnn_params_wide,
    "best_val_pr_auc": float(gnn_study_wide.best_value),
    "pr_auc_temporal": float(pr_auc_gnn_graphsage_wide_tuned),
    "change_vs_phase1b": float(graphsage_wide_change),
    "verdict": graphsage_wide_verdict,
    "at_boundary_again": wide_at_boundary_again,
})


In [ ]:
# Phase 1B only tuned GraphSAGE -- GAT was trained once with fixed defaults (hidden=64, heads=4,
# dropout=0.3, lr=0.01, 40 epochs). So there is no prior GAT search to check for a boundary; this
# search starts from a reasonably wide space directly.
def gat_objective(trial):
    hidden_dim = trial.suggest_categorical("hidden_dim", [32, 64, 128])
    heads = trial.suggest_categorical("heads", [2, 4, 8])
    lr = trial.suggest_float("lr", 1e-3, 1e-1, log=True)
    dropout = trial.suggest_float("dropout", 0.1, 0.6)
    max_epochs = trial.suggest_int("max_epochs", 20, 100)

    torch.manual_seed(SEED)
    model = GAT(in_channels=n_features, hidden_channels=hidden_dim, heads=heads, dropout=dropout).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    _, best_val = train_model_full_batch(
        model, optimizer, epochs=max_epochs, val_mask=val_mask_temporal_d, patience=10
    )
    return best_val

gat_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
gat_study.optimize(gat_objective, n_trials=WIDE_N_TRIALS, show_progress_bar=False)
print("best GAT params:", gat_study.best_params)
print("best val (steps 31-35) PR-AUC during search:", gat_study.best_value)


In [ ]:
best_gat_params = gat_study.best_params
torch.manual_seed(SEED)
best_gat_model = GAT(
    in_channels=n_features, hidden_channels=best_gat_params["hidden_dim"],
    heads=best_gat_params["heads"], dropout=best_gat_params["dropout"],
).to(DEVICE)
best_gat_optimizer = torch.optim.Adam(best_gat_model.parameters(), lr=best_gat_params["lr"], weight_decay=5e-4)
train_model_full_batch(
    best_gat_model, best_gat_optimizer, epochs=FINAL_MAX_EPOCHS,
    val_mask=val_mask_temporal_d, patience=15, verbose=True,
)
pr_auc_gat_tuned = evaluate_model(best_gat_model, test_mask_temporal_d)

gat_tuned_change = pr_auc_gat_tuned - pr_auc_gat_temporal
gat_tuned_verdict = (
    f"meaningfully changed ({gat_tuned_change:+.4f})" if abs(gat_tuned_change) > 0.01
    else f"plateaued ({gat_tuned_change:+.4f}) -- tuning barely moved GAT"
)
print(f"GAT tuned PR-AUC: {pr_auc_gat_tuned:.4f} (vs. Phase 1B untuned default: {pr_auc_gat_temporal:.4f}) "
      f"-- {gat_tuned_verdict}")

save_phase1c_progress("wider_search_gat", {
    "note": "GAT had no Optuna search in Phase 1B; this is its first tuning pass, compared against the untuned default.",
    "n_trials": WIDE_N_TRIALS,
    "best_params": best_gat_params,
    "best_val_pr_auc": float(gat_study.best_value),
    "pr_auc_temporal": float(pr_auc_gat_tuned),
    "change_vs_phase1b_untuned": float(gat_tuned_change),
    "verdict": gat_tuned_verdict,
})


## 17. Save results (Phase 1 + 1B + 1C, all numbers)

In [ ]:
if gap > 0:
    temporal_vs_random_finding = (
        f"GraphSAGE (default) scores {pr_auc_gnn_random:.4f} PR-AUC under a random split vs "
        f"{pr_auc_gnn_temporal:.4f} under the realistic temporal split (gap of {gap:+.4f}); "
        "the random split leaks future/adjacent-time information into training and so "
        "overstates real-world detection performance."
    )
else:
    temporal_vs_random_finding = (
        f"GraphSAGE (default) scores {pr_auc_gnn_temporal:.4f} PR-AUC under the temporal split vs "
        f"{pr_auc_gnn_random:.4f} under a random split (gap of {-gap:+.4f} in the temporal split's favor); "
        "reported as-is."
    )

# --- Phase 1B final paragraph: derived from whichever way the numbers land, not fixed in advance. ---
model_scores = {
    "xgboost_tier1_only": pr_auc_baseline_temporal,
    "graphsage_default": pr_auc_gnn_temporal,
    "graphsage_tuned": pr_auc_gnn_graphsage_tuned,
    "gat": pr_auc_gat_temporal,
    "xgboost_tier1_tier2": pr_auc_baseline_tier12_temporal,
}
gnn_family_best = max(pr_auc_gnn_graphsage_tuned, pr_auc_gat_temporal)

if pr_auc_baseline_tier12_temporal >= gnn_family_best:
    final_paragraph = (
        f"On this temporal split, neither GraphSAGE (tuned: {pr_auc_gnn_graphsage_tuned:.4f}) nor GAT "
        f"({pr_auc_gat_temporal:.4f}) beats a plain tuned XGBoost on raw features ({pr_auc_baseline_temporal:.4f}); "
        f"adding classical graph features (degree, out/in-degree ratio, PageRank, clustering) closes "
        f"essentially the whole gap and then some, at {pr_auc_baseline_tier12_temporal:.4f} PR-AUC. "
        "On Elliptic, under a realistic time-based evaluation, graph *structure* (via message-passing GNNs) "
        "does not earn its complexity here -- graph-derived tabular features captured with a much simpler "
        "and cheaper model do the same job or better."
    )
else:
    winner_name, winner_score = max(model_scores.items(), key=lambda kv: kv[1])
    final_paragraph = (
        f"On this temporal split, the best GNN ({winner_name.replace('_', ' ')}, {winner_score:.4f} PR-AUC) "
        f"beats both the raw-feature XGBoost baseline ({pr_auc_baseline_temporal:.4f}) and the Tier-1+Tier-2 "
        f"graph-feature XGBoost ablation ({pr_auc_baseline_tier12_temporal:.4f}). Graph structure captured "
        "through message passing carries real signal here beyond what a handful of classical graph "
        "statistics can summarize as flat features."
    )

print("Phase 1B finding:", final_paragraph)

# --- Phase 1C closing paragraph: does anything in 1C overturn the 1B headline? ---
all_gnn_scores = {
    "graphsage_tuned_phase1b": pr_auc_gnn_graphsage_tuned,
    "graphsage_tuned_wider_search": pr_auc_gnn_graphsage_wide_tuned,
    "gat_untuned_phase1b": pr_auc_gat_temporal,
    "gat_tuned_phase1c": pr_auc_gat_tuned,
    "gin": pr_auc_gin_temporal,
}
best_gnn_any_name, best_gnn_any_score = max(all_gnn_scores.items(), key=lambda kv: kv[1])
gnn_gap_to_tier12 = best_gnn_any_score - pr_auc_baseline_tier12_temporal

if best_gnn_any_score >= pr_auc_baseline_tier12_temporal:
    phase1c_closing_paragraph = (
        f"Phase 1C overturns the Phase 1B headline: {best_gnn_any_name} reaches {best_gnn_any_score:.4f} PR-AUC, "
        f"matching or beating the XGBoost Tier-1+Tier-2 ablation ({pr_auc_baseline_tier12_temporal:.4f}; "
        f"{robustness_mean:.4f} +/- {robustness_std:.4f} across seeds). With a third architecture and/or a bigger "
        "search, message passing does earn its complexity on Elliptic after all. Caveat: this is a single-seed "
        "GNN number against a 3-seed XGBoost number, so it needs its own robustness check before being claimed."
    )
else:
    phase1c_closing_paragraph = (
        f"Phase 1C does not change the Phase 1B headline -- it holds up under more scrutiny. A third architecture "
        f"(GIN: {pr_auc_gin_temporal:.4f}), a 50-trial wider search for GraphSAGE "
        f"({pr_auc_gnn_graphsage_wide_tuned:.4f}; {graphsage_wide_verdict}), and GAT's first real tuning pass "
        f"({pr_auc_gat_tuned:.4f}; {gat_tuned_verdict}) leave the best message-passing GNN at "
        f"{best_gnn_any_score:.4f} PR-AUC ({best_gnn_any_name}), {gnn_gap_to_tier12:+.4f} behind XGBoost on "
        f"raw + classical graph features ({pr_auc_baseline_tier12_temporal:.4f}; {robustness_mean:.4f} +/- "
        f"{robustness_std:.4f} across seeds). Explainability, best GNN ({best_gnn_for_explain_name}): "
        f"{best_gnn_explanation_summary} Attention vs. GNNExplainer: {gnn_explainer_vs_attention_finding} "
        f"Camouflage: {camouflage_finding} {label_coverage_note}"
    )

print("\nPhase 1C closing paragraph:", phase1c_closing_paragraph)

results = {
    # ---------------- Phase 1 ----------------
    "pr_auc_baseline_xgboost_tier1_temporal": float(pr_auc_baseline_temporal),
    "best_xgb_tier1_params": best_params,

    "pr_auc_gnn_graphsage_default_temporal": float(pr_auc_gnn_temporal),
    "pr_auc_gnn_graphsage_random": float(pr_auc_gnn_random),
    "temporal_vs_random_gap_graphsage_default": float(gap),
    "temporal_vs_random_finding": temporal_vs_random_finding,

    # ---------------- Phase 1B ----------------
    "graphsage_default_convergence_verdict": convergence_verdict,
    "graphsage_default_still_dropping": bool(still_dropping),

    "pr_auc_gnn_graphsage_tuned_temporal": float(pr_auc_gnn_graphsage_tuned),
    "graphsage_tuned_best_params": best_gnn_params,

    "pr_auc_gat_temporal": float(pr_auc_gat_temporal),
    "gat_attention_explanation_example": gat_explanation,

    "pr_auc_baseline_xgboost_tier1_tier2_temporal": float(pr_auc_baseline_tier12_temporal),
    "xgboost_tier1_tier2_best_params": tier2_best_params,

    "robustness_check": {
        "best_model": best_model_name,
        "seeds": SEEDS_FOR_ROBUSTNESS,
        "scores": robustness_scores,
        "mean": robustness_mean,
        "std": robustness_std,
    },

    "final_paragraph": final_paragraph,

    # ---------------- Phase 1C ----------------
    "phase1c": json.loads(PHASE1C_PARTIAL_PATH.read_text(encoding="utf-8")),  # gin, gnn_explainer, camouflage_check, wider_search_*
    "phase1c_summary": {
        "pr_auc_gin_temporal": float(pr_auc_gin_temporal),
        "pr_auc_gnn_graphsage_wider_search_temporal": float(pr_auc_gnn_graphsage_wide_tuned),
        "pr_auc_gat_tuned_temporal": float(pr_auc_gat_tuned),
        "camouflage_model": best_gnn_for_camo_name,
        "camouflage_definitions": {
            name: {
                "camouflaged_recall": d["camouflaged"]["recall"],
                "non_camouflaged_recall": d["non_camouflaged"]["recall"],
                "camouflaged_pr_auc_vs_licit": d["camouflaged"]["pr_auc_vs_licit"],
                "non_camouflaged_pr_auc_vs_licit": d["non_camouflaged"]["pr_auc_vs_licit"],
                "n_camouflaged": d["camouflaged"]["n"],
                "n_non_camouflaged": d["non_camouflaged"]["n"],
            }
            for name, d in camouflage_results["definitions"].items()
        },
        "best_gnn_any": best_gnn_any_name,
        "best_gnn_any_pr_auc": float(best_gnn_any_score),
    },
    "phase1c_closing_paragraph": phase1c_closing_paragraph,

    "smoke_test": SMOKE_TEST,
}

results_path = OUTPUT_DIR / "results.json"
results_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"\nwrote {results_path}")
print(json.dumps(results, indent=2))

for fname, obj in [
    ("graphsage_weights.pt", gnn),
    ("graphsage_tuned_weights.pt", best_gnn_model),
    ("gat_weights.pt", gat_model),
    ("gin_weights.pt", gin_model),
    ("graphsage_wider_search_weights.pt", best_gnn_model_wide),
    ("gat_tuned_weights.pt", best_gat_model),
]:
    torch.save(obj.state_dict(), OUTPUT_DIR / fname)
    print(f"wrote {OUTPUT_DIR / fname}")

baseline_model.save_model(str(OUTPUT_DIR / "xgboost_baseline.json"))
tier12_model.save_model(str(OUTPUT_DIR / "xgboost_tier1_tier2.json"))
print(f"wrote {OUTPUT_DIR / 'xgboost_baseline.json'}")
print(f"wrote {OUTPUT_DIR / 'xgboost_tier1_tier2.json'}")

# Written separately because this Kaggle runtime has no copy of the repo's docs/design_decisions.md --
# merge by hand into docs/design_decisions.md.
phase1c_addendum = f"""

## Phase 1C additions

- **Third architecture (task 1): GIN.** Elliptic edges are untyped, so RGCN would collapse to a
  single-relation GCN, and GCN's normalized mean aggregation is too close to GraphSAGE. GIN (sum
  aggregation + MLP, BatchNorm in the MLPs per the original paper) is a genuinely different aggregation
  family. Temporal-test PR-AUC: {pr_auc_gin_temporal:.4f} (GraphSAGE tuned {pr_auc_gnn_graphsage_tuned:.4f},
  GAT {pr_auc_gat_temporal:.4f}, XGBoost Tier-1 {pr_auc_baseline_temporal:.4f}).
- **GNNExplainer (task 2):** best GNN so far was {best_gnn_for_explain_name}; explained node
  {gnn_explainer_best['target_node']} ({gnn_explainer_best['target_selection']}). Removing all its edges changed
  the prediction by {gnn_explainer_best['prob_change_from_edge_removal']:+.4f}; feature-mask share local
  {gnn_explainer_best['feature_mask_share_local']:.0%} / neighborhood-aggregate
  {gnn_explainer_best['feature_mask_share_neighborhood_aggregate']:.0%}.
  Attention vs. GNNExplainer on GAT, node {target_node}: {gnn_explainer_vs_attention_finding}
- **Camouflage check (task 3), model {best_gnn_for_camo_name}:** {camouflage_finding}
- **Bigger search (task 4):** GraphSAGE Phase 1B best hidden_dim was at the search boundary: {was_at_boundary}
  (space used: {WIDE_HIDDEN_OPTIONS}); 50 trials -> {pr_auc_gnn_graphsage_wide_tuned:.4f} ({graphsage_wide_verdict}).
  GAT first tuning pass, 50 trials -> {pr_auc_gat_tuned:.4f} ({gat_tuned_verdict}).

### Closing paragraph (Phase 1C)

{phase1c_closing_paragraph}
"""
addendum_path = DOCS_DIR / "design_decisions_phase1c_addendum.md"
addendum_path.write_text(phase1c_addendum, encoding="utf-8")
print(f"wrote {addendum_path} -- merge into the repo's docs/design_decisions.md")


### Definition of done -- checklist for this notebook's run

**Phase 1**
- [ ] Ran with `SMOKE_TEST = False` on Kaggle (GPU on, internet on)
- [ ] `docs/graph_data_exploration.md` written with real EDA numbers
- [ ] Three original PR-AUC numbers in `artifacts/graph/results.json` (baseline temporal, GNN temporal, GNN random)

**Phase 1B**
- [ ] Convergence verdict, tuned GraphSAGE, GAT + attention example, Tier-1+Tier-2 ablation, 3-seed robustness check
- [ ] `results.json["final_paragraph"]` present

**Phase 1C**
- [ ] GIN temporal PR-AUC (`results.json["phase1c_summary"]["pr_auc_gin_temporal"]`)
- [ ] GNNExplainer on the best GNN + on GAT at the Phase 1B node, with the edge-ablation check (`results.json["phase1c"]["gnn_explainer"]`)
- [ ] Camouflage check, both definitions, recall + PR-AUC with chance baselines (`results.json["phase1c"]["camouflage_check"]`)
- [ ] 50-trial wider GraphSAGE search (boundary check result recorded) and GAT's first tuning pass
- [ ] `results.json["phase1c_closing_paragraph"]` present

**If the run dies part-way through Phase 1C**
- [ ] Download `artifacts/graph/results_phase1c_partial.json` -- it holds every Phase 1C task that finished before the crash

**To do after the Kaggle run finishes**
- [ ] Download `artifacts/graph/*` (results.json, results_phase1c_partial.json, all `.pt`/`.json` model files, loss curve png)
- [ ] Download `docs/design_decisions_phase1c_addendum.md` (or merge the save cell's output by hand)
